In [2]:
import osmium as osm
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import numpy as np
import sqlite3 as sql
import ast
np.set_printoptions(legacy='1.25')


In [3]:
class OSMHandler(osm.SimpleHandler):
    def __init__(self, filter_tags=None):
        osm.SimpleHandler.__init__(self) 
        self.nodes = []
        self.ways = []
        self.relations = []
        self.filter_tags = filter_tags or {}
        self.node_cache = {}
    
    def node(self, n):
        tags = {tag.k: tag.v for tag in n.tags}

        self.nodes.append({
            'id': n.id,
            'latitude': n.location.lat,
            'longitude': n.location.lon
            
        })

        self.node_cache[n.id] = (n.location.lon, n.location.lat)
    
    def way(self, w):
        # Highway types suitable for car navigation
        KEEP_TAGS = {'highway', 'oneway', 'name', 'maxspeed', 'junction'}
        CAR_HIGHWAY_TYPES = {
            # Major highways
            'motorway', 'motorway_link',
            'trunk', 'trunk_link', 
            'primary', 'primary_link',
            'secondary', 'secondary_link',
            'tertiary', 'tertiary_link',
            
            # Local roads
            'residential', 'unclassified', 
            'living_street', 'service',
            
            # Unknown but potentially driveable
            'road'

            # Roundabouts
            'mini_roundabout', 'turning_circle',
            'turning_loop'
            }

        tags = {tag.k: tag.v for tag in w.tags}

        highway_type = tags.get('highway')

        if highway_type not in CAR_HIGHWAY_TYPES:
            return
        
        if len(w.nodes) >= 2:
            coords = []
            node_refs = []

            for node in w.nodes:
                node_refs.append(node.ref)
                
                if node.ref in self.node_cache:
                    coords.append(self.node_cache[node.ref])
                
            if len(coords) >= 2:

                filtered_tags = {k: v for k, v in tags.items() if k in KEEP_TAGS}
                
                self.ways.append({
                    'id': w.id,
                    'node_refs': node_refs,
                    **filtered_tags
                })

way_filter = {'highway': True}  


handler = OSMHandler(way_filter)
handler.apply_file("beds.pbf")

print(f"Nodes parsed: {len(handler.nodes)}")
print(f"Ways parsed: {len(handler.ways)}")
print()
nodes_df = pd.DataFrame(handler.nodes)
ways_df = pd.DataFrame(handler.ways)

# Convert node_refs strings back to lists

# Get all node IDs used in ways
used_node_ids = set()
for refs in ways_df['node_refs']:
    used_node_ids.update(refs)

filtered_node_ids = set(nodes_df['id'])

missing_nodes = used_node_ids - filtered_node_ids
extra_nodes = filtered_node_ids - used_node_ids

print(f"Missing nodes (in ways but not in nodes_df): {missing_nodes}")
print(f"Extra nodes (in nodes_df but not in ways): {len(extra_nodes)}")
print(f"Total nodes in nodes_df: {len(filtered_node_ids)}")
print(f"Total nodes referenced in ways: {len(used_node_ids)}")
print()
# Filter nodes_df to only keep nodes in used_node_ids
nodes_df = nodes_df[nodes_df['id'].isin(used_node_ids)]
print(f"Total nodes in nodes_df after filtering: {len(nodes_df)}")


Nodes parsed: 1263910
Ways parsed: 42910

Missing nodes (in ways but not in nodes_df): set()
Extra nodes (in nodes_df but not in ways): 1053897
Total nodes in nodes_df: 1263910
Total nodes referenced in ways: 210013

Total nodes in nodes_df after filtering: 210013


In [4]:
with pd.option_context('display.max_columns', None):
    display(nodes_df)
    display(ways_df)


,id,latitude,longitude
23,208217,52.054485,-0.688008
24,208218,52.052917,-0.683021
25,208219,52.051348,-0.678024
26,208220,52.050500,-0.675327
27,208223,52.048337,-0.668842
...,...,...,...
1263894,13440420522,51.892566,-0.449624
1263895,13440420523,51.891723,-0.448240
1263907,13440420535,51.889840,-0.443979
1263908,13440420536,51.889972,-0.444009


,id,node_refs,highway,maxspeed,name,oneway,junction
0,1151,"[366628, 366608, 1257240655, 13207605498, 1257...",residential,20 mph,Pemberley Avenue,NaN,NaN
1,1152,"[369278, 373913, 373921]",residential,20 mph,St. Georges Road,NaN,NaN
2,1153,"[369341, 6523462566, 373921, 344928540, 340784]",residential,20 mph,St. Augustines Road,NaN,NaN
3,1154,"[11111729496, 9982197298, 1371508425, 13715084...",residential,20 mph,St Alban Road,NaN,NaN
4,1155,"[380898, 11111846341, 11111846343, 11111846345...",residential,20 mph,St Edmond Road,NaN,NaN
...,...,...,...,...,...,...,...
42905,1465373883,"[6943209997, 6943209998, 802787831]",service,NaN,NaN,NaN,NaN
42906,1465373884,"[802887070, 8587317]",residential,20 mph,Cambridge Road,NaN,NaN
42907,1465385017,"[13440295141, 13440295142, 13440295147, 134402...",service,NaN,NaN,NaN,NaN
42908,1465385018,"[13440295134, 13440295145, 13440295139, 134402...",service,NaN,"free daytime parking, no overnight",NaN,NaN


In [5]:


# Connect to (or create) the database
conn = sql.connect("Database/MapData.sqlite")

# Insert nodes_df as a table
nodes_df.to_sql("nodes", conn, if_exists="replace", index=False)

# Insert ways_df as a table
ways_df['node_refs'] = ways_df['node_refs'].apply(lambda x: str(x))
ways_df.to_sql("ways", conn, if_exists="replace", index=False)

conn.close()
